# Stage 1.1 — Probability & Bayesian Inference

**Why this matters for NOA-AI:** NOA does not output yes/no for diseases. Instead it predicts the chance of disease `P(sepsis) = 0.73`. To reason about whether that number is meaningful -- or dangerously wrong -- you need to understand what probability actually is, how conditional reasoning works, and what Bayes' theorem lets you compute.

This notebook covers:
1. Core probability -- marginal, joint, conditional
2. Bayes' theorem -- flipping a conditional
3. Applied to NOA-AI -- what a neonatal sepsis prediction model is actually doing
4. The calibration connection -- why `P = 0.73` can be a lie
5. Exercise -- PPV/NPV for the a neonatal movement quality classifier

---
## Section 1 -- Core Probability

**Marginal probability** is the probability of one event on its own: P(sepsis). No conditions, no context.

**Joint probability** is the probability of two events happening together: P(sepsis AND HRV_suppressed).

**Conditional probability** is the probability of one event *given* that another has already occurred: P(sepsis | HRV_suppressed)

The formula:

$$P(A | B) = \frac{P(A \cap B)}{P(B)}$$

Read it as: the fraction of B-events that are also A-events.

Let's work this out numerically using a 2x2 contingency table -- the same structure underlying every classifier evaluation.

In [ ]:
import numpy as np

# Hypothetical cohort of 1000 NICU admissions
# Rows: actual sepsis status. Columns: HRV suppression detected.
#
#                HRV suppressed    HRV normal    | Total
#  Sepsis             85               15         |  100
#  No sepsis         171              729         |  900
#  Total             256              744         | 1000

table = np.array([[85, 15],
                  [171, 729]], dtype=float)
N = table.sum()

# Marginal probabilities
P_sepsis         = table[0, :].sum() / N   # P(sepsis)
P_hrv_suppressed = table[:, 0].sum() / N   # P(HRV suppressed)

# Joint probability
P_both = table[0, 0] / N  # P(sepsis AND HRV suppressed)

# Conditional probability
P_sepsis_given_hrv = P_both / P_hrv_suppressed  # P(sepsis | HRV suppressed)

print(f'P(sepsis)                    = {P_sepsis:.3f}')
print(f'P(HRV suppressed)            = {P_hrv_suppressed:.3f}')
print(f'P(sepsis AND HRV suppressed) = {P_both:.3f}')
print(f'P(sepsis | HRV suppressed)   = {P_sepsis_given_hrv:.3f}')
print()
print(f'Without the HRV signal, P(sepsis) = {P_sepsis:.0%}.')
print(f'With HRV suppression, P(sepsis) jumps to {P_sepsis_given_hrv:.1%} -- that is the diagnostic value of the signal.')

In [ ]:
# apply the same calculation to a different signal.
#
# In the same 1000-patient cohort, temperature instability was also recorded:
#
#                   Temp unstable    Temp stable    | Total
#  Sepsis                60               40         |  100
#  No sepsis             90              810         |  900
#  Total                150              850         | 1000

table_temp = np.array([[60, 40],
                       [90, 810]], dtype=float)
N_temp = table_temp.sum()

# Fill in each ???. Don't look back at the worked example.
P_sepsis_t       = ???   # all sepsis rows / total
P_temp_unstable  = ???   # all temp-unstable column / total
P_both_t         = ???   # cell [0,0] / total
P_sepsis_given_T = ???   # P_both_t / P_temp_unstable

print(f'P(sepsis):                       {P_sepsis_t:.3f}')
print(f'P(temp unstable):                {P_temp_unstable:.3f}')
print(f'P(sepsis AND temp unstable):     {P_both_t:.3f}')
print(f'P(sepsis | temp unstable):       {P_sepsis_given_T:.3f}')
print()
# Before running: compared to the HRV signal (33.2% posterior), is temperature instability
# a stronger or weaker diagnostic signal for sepsis? Can you tell from the table before computing?

---
## Section 2 -- Bayes' Theorem

The problem: in clinical practice, you *see* the symptom, and you want to know the disease probability. But your test was characterized the other way around -- you know P(HRV_suppressed | sepsis), not P(sepsis | HRV_suppressed).

Bayes' theorem lets you flip a conditional:

$$P(H | E) = \frac{P(E | H) \cdot P(H)}{P(E)}$$

Name the pieces:
- **Prior** -- P(H): what you believed before seeing the evidence. For sepsis: the background rate in your NICU.
- **Likelihood** -- P(E | H): how probable is the evidence if the hypothesis is true? P(HRV_suppressed | sepsis).
- **Evidence** -- P(E): how often does the evidence appear overall? P(HRV_suppressed) across both sick and healthy babies.
- **Posterior** -- P(H | E): what you believe *after* seeing the evidence. P(sepsis | HRV_suppressed).

The denominator P(E) expands via the law of total probability: P(E) = P(E|H)*P(H) + P(E|not H)*P(not H).

In [ ]:
# Your turn — compute the posterior for a different NOA module.
#
# The skin perfusion sensor (used for NEC detection) has these test characteristics:
#   Sensitivity: 0.72   P(perfusion_signal | NEC)
#   Specificity: 0.88   P(no signal | no NEC)
#   Prevalence:  0.05   NEC affects ~5% of NICU admissions
#
# Compute P(NEC | perfusion signal positive).
# Same three steps: fpr → P_e → posterior.

sensitivity_nec = ???
specificity_nec = ???
prevalence_nec  = ???

fpr_nec      = ???   # 1 - specificity
P_e_nec      = ???   # sensitivity * prev + fpr * (1 - prev)
posterior_nec = ???  # (sensitivity * prev) / P_e

print(f'Prior P(NEC):                   {prevalence_nec:.2f}')
print(f'Posterior P(NEC | perfusion+):  {posterior_nec:.3f}')
print()
# Compare to the HRV sepsis result: 10% prior → 33.2% posterior.
# This sensor has a lower sensitivity — does the posterior move more or less? Why?

**Before running the next cell — make a prediction.**

The chart will plot P(sepsis | HRV suppressed) across prevalences from 1% to 30%, using sensitivity=0.85 and specificity=0.81.

Write down your answers before running:
1. At 10% prevalence, what do you expect the posterior to be? (You computed it in Section 2.)
2. Roughly at what prevalence does the posterior cross 50%?
3. Will the curve be linear or curved? Why?

_(Write your predictions here, then run and compare.)_

In [ ]:
# Quick check: compute the posterior at 3 specific prevalences manually.
# Write out the formula directly — no loop. Then verify your numbers against the chart above.

sensitivity = 0.85
specificity = 0.81
fpr = 1 - specificity

# At 5% prevalence:
p_e_5  = ???
post_5 = ???

# At 15% prevalence:
p_e_15  = ???
post_15 = ???

# At 25% prevalence:
p_e_25  = ???
post_25 = ???

print(f'5%  prevalence  ->  P(sepsis | HRV+) = {post_5:.1%}')
print(f'15% prevalence  ->  P(sepsis | HRV+) = {post_15:.1%}')
print(f'25% prevalence  ->  P(sepsis | HRV+) = {post_25:.1%}')
print()
# Do these match the chart? If any are off, check your formula against Section 2.

In [ ]:
# Bayes' theorem from clinical test characteristics

sensitivity = 0.85   # P(HRV suppressed | sepsis)   -- true positive rate
specificity = 0.81   # P(HRV normal | no sepsis)    -- true negative rate
prevalence  = 0.10   # P(sepsis)                    -- prior, background rate in NICU

fpr = 1 - specificity             # P(HRV suppressed | no sepsis)
P_e = sensitivity * prevalence + fpr * (1 - prevalence)  # law of total probability

posterior = (sensitivity * prevalence) / P_e  # P(sepsis | HRV suppressed)

print(f'Prior P(sepsis):                  {prevalence:.2f}')
print(f'Likelihood P(HRV+ | sepsis):      {sensitivity:.2f}')
print(f'P(HRV suppressed) overall:        {P_e:.3f}')
print(f'Posterior P(sepsis | HRV+):       {posterior:.3f}')
print()
print(f'A positive HRV signal moves sepsis probability from {prevalence:.0%} to {posterior:.1%}.')
print('That is Bayes updating a prior with evidence.')

**Before running the calibration plot — make a prediction.**

The overconfident model outputs these predicted vs. actual rates (from the code below):

| Predicted | Actual |
|-----------|--------|
| 0.50      | 0.28   |
| 0.60      | 0.38   |
| 0.70      | 0.45   |
| 0.80      | 0.52   |

If a clinician sets a NOA alert threshold at "fire when P(sepsis) > 0.65", what is the actual sepsis rate among patients who trigger that alert? Write your answer before running.

Your answer: _______

Why does this matter for the three-tier alert thresholds in NOA-SEPSIS-001?

---
## Section 3 -- Applied to NOA-AI (M06 Sepsis Predictor)

M06 does not apply Bayes' theorem manually -- it learns the mapping from sensor inputs to `P(sepsis)` from data, using the HeRO (Heart Rate Observation) paradigm. But the *structure* of what it is doing is Bayesian:

- **Prior:** background sepsis rate in the NICU (roughly 5-10% in Level III/IV settings)
- **Evidence:** HRV suppression, temperature instability, perfusion changes -- from surface-EMG-sensor sensors
- **Posterior:** `P(sepsis | HRV_drop, temp_instability, ...)` -- the output the model produces

The HeRO paradigm exploits the fact that HRV characteristic changes precede clinical sepsis by 12-24 hours. The model learns: given this HRV pattern, what is P(sepsis onset within 24h)?

**The critical insight: base rate (prevalence) matters enormously.** The same sensor signal produces a very different posterior depending on prevalence. Let's show that.

In [ ]:
# a neonatal movement quality classifier — PPV and NPV at varying CP prevalence
# Sensitivity = 0.98, Specificity = 0.91
#
# PPV = TP / (TP + FP)   of all positive tests, how many are true positives?
# NPV = TN / (TN + FN)   of all negative tests, how many are true negatives?
#
# At each prevalence:
#   TP  = sensitivity * prevalence
#   FP  = (1 - specificity) * (1 - prevalence)
#   TN  = specificity * (1 - prevalence)
#   FN  = (1 - sensitivity) * prevalence

sensitivity = 0.98
specificity = 0.91

print(f'{"Prevalence":>12} {"PPV":>8} {"NPV":>8}')
print('-' * 32)

for prev in [0.01, 0.05, 0.10]:
    tp = ???   # sensitivity * prev
    fp = ???   # (1 - specificity) * (1 - prev)
    tn = ???   # specificity * (1 - prev)
    fn = ???   # (1 - sensitivity) * prev

    ppv = ???  # TP / (TP + FP)
    npv = ???  # TN / (TN + FN)

    print(f'{prev:>11.0%} {ppv:>8.1%} {npv:>8.1%}')

**After computing:** Why does M05 target 98% sensitivity even though PPV at 1% prevalence is only ~10%?

Write your answer here before reading on. Think about:
- What happens to a baby when M05 misses a CP case (false negative)?
- What happens when M05 incorrectly flags a healthy baby (false positive)?
- Which type of error is harder to fix after the fact?

_(Your answer here.)_

---

**Reference answer** — don't read until you've written yours:

A missed CP case means a child doesn't get early physiotherapy intervention — a window that closes between 9 and 20 weeks post-term and can't be recovered. A false positive triggers a referral that a follow-up assessment resolves at low cost. The asymmetry of harm drives the threshold: one error is irreversible, the other is correctable. This is why NOA's alert tiers are tuned to minimize false negatives, not to maximize raw accuracy or PPV.

The same HRV alert carries very different clinical weight at 5% vs 20% prevalence. This is why NOA needs calibrated thresholds per deployment context, and why LMIC settings may require different alert thresholds than Canadian NICUs.

Reference: NOA-SEPSIS-001 -- three-tier alert thresholds are calibrated to minimize false negatives. Those thresholds implicitly encode an assumption about prevalence.

---
## Section 4 -- The Calibration Connection

NOA outputs `P(sepsis) = 0.73`. What does that mean?

**If the model is calibrated:** of all patients where NOA outputs approximately 0.73, roughly 73% actually develop sepsis. The number is trustworthy.

**If the model is miscalibrated:** the model consistently outputs 0.73 when the true rate is 30% -- or 90%. The number sounds confident but is wrong. A clinician who trusts the probability to set their alert threshold will act on false information.

A **reliability diagram** (calibration curve) plots predicted probability on the x-axis against actual positive rate on the y-axis. A perfectly calibrated model falls on the diagonal.

This is what Stage 6.1 builds on. Planting the seed here.

In [ ]:
# Toy reliability diagram: calibrated vs. overconfident model

predicted_probs     = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
calibrated_actual   = np.array([0.10, 0.19, 0.31, 0.39, 0.51, 0.61, 0.69, 0.81, 0.88])
overconfident_actual = np.array([0.05, 0.09, 0.14, 0.20, 0.28, 0.38, 0.45, 0.52, 0.60])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(predicted_probs, calibrated_actual,    'o-', color='steelblue', label='Calibrated model')
ax.plot(predicted_probs, overconfident_actual, 's-', color='coral',     label='Overconfident model')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Actual positive rate')
ax.set_title('Reliability diagram\nOverconfident model: P=0.73 but only 45% actually have sepsis')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Overconfident model at P=0.7: actual rate ~45%, not 70%.')
print('A clinician who sets alerts at P > 0.65 expecting 65%+ probability')
print('is actually acting on cases with ~40% probability. The threshold is wrong.')

---
## Section 5 -- Exercise: PPV and NPV for M05

The a neonatal movement quality classifier has specified performance: **98% sensitivity, 91% specificity**.

Two more quantities matter clinically:
- **PPV (Positive Predictive Value):** P(CP | test positive) -- if M05 flags abnormal movement, how likely is CP?
- **NPV (Negative Predictive Value):** P(no CP | test negative) -- if M05 gives a normal result, how confident can you be?

Both depend on prevalence. Compute PPV and NPV at CP prevalences of 1%, 5%, and 10%. Then answer: why does M05 target 98% sensitivity even though that produces false positives?

In [ ]:
# a neonatal movement quality classifier -- PPV and NPV at varying CP prevalence

sensitivity = 0.98
specificity = 0.91
fpr = 1 - specificity
fnr = 1 - sensitivity

print(f'{"Prevalence":>12} {"PPV":>8} {"NPV":>8}')
print('-' * 32)

for prev in [0.01, 0.05, 0.10]:
    tp = sensitivity * prev
    fp = fpr * (1 - prev)
    tn = specificity * (1 - prev)
    fn = fnr * prev

    ppv = tp / (tp + fp)
    npv = tn / (tn + fn)

    print(f'{prev:>11.0%} {ppv:>8.1%} {npv:>8.1%}')

print()
print('Why 98% sensitivity even at low PPV?')
print()
print('A missed CP case (false negative) means a child does not get early intervention --')
print('a window that closes between 9-20 weeks post-term. A false positive triggers a')
print('referral that a follow-up assessment resolves. The asymmetry of harm drives the')
print('threshold. This is why NOA alert tiers minimize false negatives, not maximize accuracy.')

---
## Summary

- **Conditional probability** is the core operation: P(disease | evidence), computed from the 2x2 table
- **Bayes' theorem** lets you compute the posterior from a prior and a likelihood -- the structure underlying every probabilistic classifier
- **Base rate dominates**: the same test gives very different posteriors at different prevalence levels
- **Calibration**: a model's output probability is only meaningful if calibrated. If it isn't, P=0.73 is just a number, not a probability
- **PPV/NPV vs. sensitivity/specificity**: sensitivity and specificity are properties of the test; PPV and NPV depend on the population being tested. In screening at low prevalence, even a high-sensitivity test has low PPV

**Next:** Stage 1.2 -- Classification & Binary Outcomes (`02-classification-outcomes.ipynb`)

---
## Results

Key numerical findings from the code cells above:

| Computation | Result |
|---|---|
| P(sepsis \| HRV suppressed), 10% base rate | **33.2%** — vs 10% prior (3.3× lift) |
| P(sepsis \| HRV suppressed), 20% base rate | **53%** |
| Movement classifier PPV at 1% CP prevalence | **9.9%** |
| Movement classifier PPV at 5% CP prevalence | **36.4%** |
| Movement classifier PPV at 10% CP prevalence | **54.7%** |
| Movement classifier NPV across 1–10% prevalence | **≥99.8%** |
| Overconfident model at predicted P = 0.70 | actual rate **~45%** — a 25-point gap |

**The base-rate effect is the headline result.** The same HRV test (sensitivity 0.85, specificity 0.81) lifts sepsis probability from 10% to 33% at typical NICU prevalence — but the jump is much smaller at lower base rates. Alert thresholds calibrated on one deployment context can be dangerously wrong in another.

**Why 98% sensitivity at 9.9% PPV?** At 1% CP prevalence, only 1 in 10 positive flags is a true case — but missing a true case forecloses an irreversible early-intervention window. A false positive triggers a referral that follow-up resolves. The asymmetry of harm, not raw accuracy, drives the threshold design.